## NEKT SDK SETUP

In [1]:
import nekt
import os
from   dotenv import load_dotenv

load_dotenv()
nekt.data_access_token  = os.getenv("NEKT_DATA_ACCESS_TOKEN")
nekt.engine             = "spark"

## IMPORTS

In [2]:
from typing         import Optional
from pyspark.sql    import Window
from pyspark.sql    import DataFrame, Column
from pyspark.sql    import functions as F

## HELPER FUNCTIONS

In [3]:
def extract_nekt_table(layer_name: str, table_name: str) -> DataFrame:
    """Simplify nekt table extraction syntax."""
    return nekt.load_table(layer_name=layer_name, table_name=table_name)

def save_nekt_table(
    df: DataFrame,
    layer_name: str,
    table_name: str,
    folder_name: Optional[str] = None
):
    """Simplify nekt table saving syntax."""
    nekt.save_table(
        df=df,
        layer_name=layer_name,
        table_name=table_name,
        folder_name=folder_name
    )

def ms_to_timestamp(col_name: str) -> Column:
    """Converts a Unix millisecond epoch column to a TimestampType."""
    return F.to_timestamp(F.col(col_name).cast("long") / 1000)

def last_element(array_col: str, field: str) -> F.Column:
    """Safely retrieves a field from the last element of an array column."""
    return (
        F.when(
            F.size(F.col(array_col)) > 0,
            F.element_at(F.col(array_col), F.size(F.col(array_col)))[field]
        ).otherwise(F.lit(None))
    )

## EXTRACTING TABLES

In [4]:
# pipedrive - bronze tables
df_bronze_deals         = extract_nekt_table("Bronze", "bunzl_pipedrive_bronze_deals")
df_bronze_organizations = extract_nekt_table("Bronze", "bunzl_pipedrive_bronze_organizations")
df_bronze_pipelines     = extract_nekt_table("Bronze", "bunzl_pipedrive_bronze_pipelines")
df_bronze_stages        = extract_nekt_table("Bronze", "bunzl_pipedrive_bronze_stages")
df_bronze_users         = extract_nekt_table("Bronze", "bunzl_pipedrive_bronze_users")

:: loading settings :: url = jar:file:/usr/local/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
com.google.cloud.spark#spark-bigquery-with-dependencies_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7baa37ef-a037-4050-81d8-9fd2d739ec50;1.0
	confs: [default]


	found io.delta#delta-spark_2.12;3.3.0 in central
	found io.delta#delta-storage;3.3.0 in central


	found org.antlr#antlr4-runtime;4.9.3 in central


	found com.google.cloud.spark#spark-bigquery-with-dependencies_2.12;0.43.1 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.3.0/delta-spark_2.12-3.3.0.jar ...


...

	[SUCCESSFUL ] com.google.cloud.spark#spark-bigquery-with-dependencies_2.12;0.43.1!spark-bigquery-with-dependencies_2.12.jar (3105ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.3.0/delta-storage-3.3.0.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.3.0!delta-storage.jar (106ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...


	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (138ms)
:: resolution report :: resolve 18204ms :: artifacts dl 4064ms
	:: modules in use:
	com.google.cloud.spark#spark-bigquery-with-dependencies_2.12;0.43.1 from central in [default]
	io.delta#delta-spark_2.12;3.3.0 from central in [default]
	io.delta#delta-storage;3.3.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   4   |   4   |   4   |   0   ||   4   |   4   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-7baa37ef-a037-4050-81d8-9fd2d739ec50
	confs: [default]
	4 artifacts copied, 0 already retrieved 

26/06/11 20:29:33 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/11 20:29:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


## TRANSFORMING TABLES

In [5]:
# pipedrive - silver data (deals enriched with org, pipeline, stage, owner)
df_silver_pipedrive_data = (
    df_bronze_deals.alias("d")
    .filter(
        F.col("d.id").isNotNull() &
        F.col("d.add_time").isNotNull() &
        (F.col("d.is_deleted") == False)
    )
    .join(
        df_bronze_organizations.alias("o"),
        on=F.col("d.org_id") == F.col("o.id"),
        how="left",
    )
    .join(
        df_bronze_pipelines.alias("p"),
        on=F.col("d.pipeline_id") == F.col("p.id"),
        how="left",
    )
    .join(
        df_bronze_stages.alias("s"),
        on=F.col("d.stage_id") == F.col("s.id"),
        how="left",
    )
    .join(
        df_bronze_users.alias("u"),
        on=F.col("d.owner_id") == F.col("u.id"),
        how="left",
    )
    .select(
        # deal - identifiers
        F.col("d.id")                        .cast("integer").alias("deal_id"),
        F.col("d.add_time")                  .cast("string") .alias("deal_add_time"),
        F.col("d.origin")                    .cast("string") .alias("deal_origin"),
        F.col("d.origin_id")                 .cast("integer").alias("deal_origin_id"),
        F.col("d.channel")                   .cast("integer").alias("deal_channel"),
        F.col("d.channel_id")                .cast("string") .alias("deal_channel_id"),
        # deal - info
        F.col("d.title")                     .cast("string") .alias("deal_title"),
        F.col("d.value")                     .cast("integer").alias("deal_value"),
        F.col("d.currency")                  .cast("string") .alias("deal_currency"),
        F.col("d.status")                    .cast("string") .alias("deal_status"),
        F.col("d.lost_reason")               .cast("string") .alias("deal_lost_reason"),
        F.col("d.probability")               .cast("integer").alias("deal_probability"),
        F.col("d.is_deleted")                .cast("boolean").alias("deal_is_deleted"),
        # deal - dates
        F.col("d.update_time")               .cast("string") .alias("deal_update_time"),
        F.col("d.stage_change_time")         .cast("string") .alias("deal_stage_change_time"),
        F.col("d.expected_close_date")       .cast("string") .alias("deal_expected_close_date"),
        F.col("d.close_time")                .cast("string") .alias("deal_close_time"),
        F.col("d.won_time")                  .cast("string") .alias("deal_won_time"),
        F.col("d.lost_time")                 .cast("string") .alias("deal_lost_time"),
        # deal - activity counts
        F.col("d.activities_count")          .cast("integer").alias("deal_activities_count"),
        F.col("d.done_activities_count")     .cast("integer").alias("deal_done_activities_count"),
        F.col("d.undone_activities_count")   .cast("integer").alias("deal_undone_activities_count"),
        # pipeline
        F.col("d.pipeline_id")               .cast("integer").alias("pipeline_id"),
        F.col("p.name")                      .cast("string") .alias("pipeline_name"),
        # stage
        F.col("d.stage_id")                  .cast("integer").alias("stage_id"),
        F.col("s.name")                      .cast("string") .alias("stage_name"),
        # owner
        F.col("d.owner_id")                  .cast("integer").alias("owner_id"),
        F.col("u.name")                      .cast("string") .alias("owner_name"),
        F.col("u.email")                     .cast("string") .alias("owner_email"),
        # organization - identifiers
        F.col("d.org_id")                    .cast("integer").alias("org_id"),
        F.col("o.name")                      .cast("string") .alias("org_name"),
        F.col("o.custom_fields").getField("e1897931095ea6e1bba9b0ebdecfb6ad7587ec27").cast("string").alias("org_cnpj"),
        # organization - address
        F.col("o.address.value")             .cast("string") .alias("org_address"),
        F.col("o.address.route")             .cast("string") .alias("org_address_route"),
        F.col("o.address.street_number")     .cast("string") .alias("org_address_street_number"),
        F.col("o.address.sublocality")       .cast("string") .alias("org_address_sublocality"),
        F.col("o.address.locality")          .cast("string") .alias("org_address_locality"),
        F.col("o.address.admin_area_level_1").cast("string") .alias("org_address_state"),
        F.col("o.address.admin_area_level_2").cast("string") .alias("org_address_city"),
        F.col("o.address.country")           .cast("string") .alias("org_address_country"),
        F.col("o.address.postal_code")       .cast("string") .alias("org_address_postal_code"),
        # audit
        F.current_timestamp()                .cast("string") .alias("_loaded_at"),
    )
    .dropDuplicates(
        ["deal_id"]
    )
)

## LOADING TABLES

In [6]:
# pipedrive - silver tables
# save_nekt_table(df_silver_pipedrive_data, "Silver", "bunzl_pipedrive_silver_data", "bunzl_pipedrive_silver")

print(f"Row count: {df_silver_pipedrive_data.count()}")
df_silver_pipedrive_data.toPandas()

Row count: 1210


,deal_id,deal_add_time,deal_origin,deal_origin_id,deal_channel,deal_channel_id,deal_title,deal_value,deal_currency,deal_status,...,org_address,org_address_route,org_address_street_number,org_address_sublocality,org_address_locality,org_address_state,org_address_city,org_address_country,org_address_postal_code,_loaded_at
0,15,2025-01-02 14:12:49,ManuallyCreated,NaN,9.0,NaN,Negócio Willian Serta Tercerização,5694,BRL,lost,...,None,None,None,None,None,None,None,None,None,2026-06-11 20:30:03.203183
1,16,2025-01-02 14:24:49,ManuallyCreated,NaN,NaN,NaN,Negócio Projeman Manutenção-Luciano,576,BRL,lost,...,None,None,None,None,None,None,None,None,None,2026-06-11 20:30:03.203183
2,17,2025-01-02 16:56:12,ManuallyCreated,NaN,NaN,NaN,Negócio Rosiane Pacheco <comercial@rssupply.on...,16048,BRL,lost,...,None,None,None,None,None,None,None,None,None,2026-06-11 20:30:03.203183
3,18,2025-01-02 17:08:19,ManuallyCreated,NaN,NaN,NaN,Negócio Adriana ITATINGA MINEGARAÇÃO,0,BRL,lost,...,None,None,None,None,None,None,None,None,None,2026-06-11 20:30:03.203183
4,19,2025-01-02 19:51:00,ManuallyCreated,NaN,NaN,NaN,Negócio Jadson SBT,4971,BRL,lost,...,None,None,None,None,None,None,None,None,None,2026-06-11 20:30:03.203183
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1205,1221,2026-06-09 18:24:08,WebForms,0.0,3.0,Unidades - Contagem,Ronaldo Gonçalves Paizante organização,0,BRL,open,...,None,None,None,None,None,None,None,None,None,2026-06-11 20:30:03.203183
1206,1222,2026-06-09 19:38:52,WebForms,0.0,3.0,Unidades - Guarulhos,CAMILA MARIA NORONHA organização,0,BRL,open,...,None,None,None,None,None,None,None,None,None,2026-06-11 20:30:03.203183
1207,1223,2026-06-09 19:39:55,WebForms,0.0,3.0,Unidades - Camaçari,Saulo Amorim organização,0,BRL,open,...,None,None,None,None,None,None,None,None,None,2026-06-11 20:30:03.203183
1208,1224,2026-06-10 14:32:36,WebForms,0.0,3.0,Unidades - Camaçari,Leonardo Vidal organização,0,BRL,open,...,None,None,None,None,None,None,None,None,None,2026-06-11 20:30:03.203183
